# Rancher Kubernetes (RKE2) on FABRIC via Ansible

This notebook provisions a **2-node** FABRIC slice on a single site and uses Ansible to deploy [RKE2](https://docs.rke2.io/) (Rancher Kubernetes Engine 2).

| Role | Node | Resources |
|------|------|-----------|
| Control plane (`rke2-server`) | `node1` | 8 cores, 16 GB RAM |
| Worker (`rke2-agent`) | `node2` | 8 cores, 16 GB RAM |

Cluster traffic uses a private L2 dataplane (`192.168.1.0/24`). Ansible reaches the nodes over the FABRIC management network.

## Import the FABlib Library

In [ ]:
from ipaddress import IPv4Network
import random
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

## Find a Site With Enough Capacity

We need **2 nodes × 8 cores × 16 GB RAM** on one site. Requirements are scaled by `1.2` so a busy site is less likely to fail mid-submit.

In [ ]:
resources = fablib.get_resources()
resources.update()

nodesReq = 2
coresReq = 8
ramReq = 16

# Scale up a bit to account for others joining the selected site.
totalCoreAvail = nodesReq * coresReq * 1.2
totalRamAvail = nodesReq * ramReq * 1.2

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(f"Need ~{totalCoreAvail:.0f} cores and ~{totalRamAvail:.0f} GB RAM available")
print(f"Usable sites ({len(usableSite)}): {usableSite}")
assert usableSite, "No site currently has enough free cores/RAM for this slice"

## Create the Slice

Site selection is random among usable sites so concurrent class users are less likely to collide.

In [ ]:
siteName = random.choice(usableSite)
sliceName = "RancherK8s"
network_name = "rke2net"
print(f"Selected site: {siteName}")

slice = fablib.new_slice(name=sliceName)
net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(
        name=f"node{i}",
        site=siteName,
        cores=coresReq,
        ram=ramReq,
        disk=50,
        image="default_ubuntu_22",
    )
    iface = node.add_component(model="NIC_Basic", name="nic").get_interfaces()[0]
    iface.set_mode("config")
    net.add_interface(iface)

slice.submit()

In [ ]:
slice.wait_ssh()

for node in slice.get_nodes():
    print("----", node.get_name(), "----")
    print("management ip:", node.get_management_ip())
    print("username:", node.get_username())
    print(node.get_ssh_command())

## Configure Dataplane Networking

Assign private IPs on the L2 network. RKE2 will advertise and join using these addresses.

In [ ]:
for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)
    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24"),
    )
    print(f"{node.get_name()} -> 192.168.1.{i}")

**Keep re-running the cell below until every node can ping every other node.**

In [ ]:
for i in range(1, nodesReq):
    src = slice.get_node(name=f"node{i}")
    for j in range(i + 1, nodesReq + 1):
        des = slice.get_node(name=f"node{j}")
        des_addr = des.get_interface(network_name=network_name).get_ip_addr()
        print(f"{src.get_name()} is pinging {des.get_name()} at {des_addr} ========")
        stdout, stderr = src.execute(f"ping -c 2 {des_addr}")

## Create Ansible Inventory

- `node1` → `rke2_servers` (control plane)
- `node2` → `rke2_agents` (worker)

A shared `rke2_token` is written into inventory so agents can join without scraping the server token file by hand.

In [ ]:
node_defs = []
for node in slice.get_nodes():
    name = node.get_name()
    private_ip = str(node.get_interface(network_name=network_name).get_ip_addr())
    group = "rke2_servers" if name == "node1" else "rke2_agents"
    node_defs.append({"name": name, "private_ip": private_ip, "group": group})

slice_key = fablib.get_default_slice_key()["slice_private_key_file"]
ssh_config = "/home/fabric/work/fabric_config/ssh_config"

for nd in node_defs:
    node = slice.get_node(nd["name"])
    stdout, stderr = node.execute(
        "python3 -c 'import sys; print(sys.executable)'",
        quiet=True,
    )
    nd["node"] = node
    nd["python"] = stdout.strip()

lines = []
lines.append("all:")
lines.append("  vars:")
lines.append("    ansible_become: true")
lines.append(f'    ansible_ssh_private_key_file: "{slice_key}"')
lines.append(f'    ansible_ssh_common_args: "-F {ssh_config}"')
lines.append('    rke2_token: "fabric-rke2-cluster-token"')
lines.append('    rke2_server_private_ip: "192.168.1.1"')
lines.append("")
lines.append("  children:")

for group in ("rke2_servers", "rke2_agents"):
    lines.append(f"    {group}:")
    lines.append("      hosts:")
    for nd in node_defs:
        if nd["group"] != group:
            continue
        node = nd["node"]
        lines.append(f'        {nd["name"]}:')
        lines.append(f'          ansible_host: "{node.get_management_ip()}"')
        lines.append(f'          ansible_user: "{node.get_username()}"')
        lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
        lines.append(f'          private_ip: "{nd["private_ip"]}"')
    lines.append("")

inventory = "\n".join(lines) + "\n"
Path("playbook").mkdir(exist_ok=True)
Path("playbook/inventory.yml").write_text(inventory)

print("Wrote playbook/inventory.yml")
print(inventory)

## Playbook: Host Prerequisites

Disables swap, loads `overlay` / `br_netfilter`, enables IP forwarding, and writes `/etc/hosts` entries for the cluster nodes.

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-prereqs.yml

## Playbook: Install RKE2

Declarative take on the [RKE2 quick start](https://docs.rke2.io/install/quickstart):

1. Write `config.yaml` so the server advertises on the dataplane IP
2. Install and start `rke2-server` on `node1`
3. Install and start `rke2-agent` on `node2` using the shared token
4. Deploy a small `nginx` Deployment + NodePort Service (`30080`) for a smoke test

First-time install commonly takes **10–15 minutes** while images are pulled.

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-rke2.yml

## Verify the Cluster

In [ ]:
server = slice.get_node("node1")

print("==== nodes ====")
stdout, stderr = server.execute("kubectl get nodes -o wide", quiet=True)
print(stdout)

print("==== nginx-demo ====")
stdout, stderr = server.execute("kubectl get pods,svc -l app=nginx-demo -o wide", quiet=True)
print(stdout)

print("==== curl via NodePort on dataplane ====")
stdout, stderr = server.execute("curl -s -o /dev/null -w '%{http_code}\n' http://192.168.1.1:30080/", quiet=True)
print("HTTP status:", stdout.strip())

## Useful Follow-ups

On `node1` (after SSH):

```bash
kubectl get pods -A
kubectl describe node node1
sudo journalctl -u rke2-server -f
```

On `node2`:

```bash
sudo journalctl -u rke2-agent -f
```

To reach the NodePort from your laptop, create an SSH tunnel to `node1:30080` (same pattern as the Docker Swarm notebook).

## Cleanup

In [ ]:
# Uncomment when you are finished
# slice.delete()